In [9]:
import tensorflow as tf
import pandas as pd
import numpy as np
import tensorflow_lattice as tfl

In [10]:
train_data = pd.read_csv('./data/train_data.csv')
val_data = pd.read_csv('./data/val_data.csv')

feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Win'  # We want to predict the margin

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_val = val_data[feature_cols]
y_val = val_data[target_col]

# For Brier score, we need the actual outcome: 1 if Team A won, 0 otherwise
y_val_binary = val_data['Win'].values

In [11]:
###############################################################################
# 5. TensorFlow Lattice (Older PWLCalibration API, custom monotonicities)
###############################################################################

# We'll define a custom Brier metric in TF
def brier_score_tf(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(tf.square(y_true - y_pred))

def build_tf_lattice_custom_monotonic_model_legacy_lattice_sizes(hp, feature_names, X_train):
    """
    Older TF Lattice code requiring 'lattice_sizes' in Lattice(...).
    We'll do:
      - For each feature, a PWLCalibration with 'input_keypoints' array (no num_keypoints param).
      - Then a Lattice layer with lattice_sizes=[2,2,...] (or [3,3,...]) plus monotonicities.
    """

    # Suppose we define some custom monotonic map:
    custom_monotonic_map = {
        'TOPerc_diff_7_A': 'decreasing',
        'TOPerc_diff_7_B': 'decreasing',
    }

    # We'll pretend we have a tuner param for how many keypoints to use in PWLCalibration
    num_keypoints = hp.Int('num_keypoints', min_value=5, max_value=15, step=5, default=10)
    lr = hp.Float('learning_rate', 1e-4, 1e-2, sampling='log', default=1e-3)

    inputs = {}
    calibrators = []
    for feat in feature_names:
        inputs[feat] = tf.keras.Input(shape=(1,), name=feat)

        # monotonic direction for PWL
        if feat in custom_monotonic_map:
            this_monotonic = custom_monotonic_map[feat]
        else:
            this_monotonic = 'increasing'

        # build input_keypoints array
        f_min = float(X_train[feat].min())
        f_max = float(X_train[feat].max())
        keypoints = np.linspace(f_min, f_max, num_keypoints)

        # PWLCalibration older signature
        c = tfl.layers.PWLCalibration(
            input_keypoints=keypoints,
            units=1,
            output_min=0.0,
            output_max=1.0,
            clamp_min=False,
            clamp_max=False,
            monotonicity=this_monotonic
        )(inputs[feat])
        calibrators.append(c)

    # Concatenate calibrator outputs
    concat_calibrators = tf.keras.layers.Concatenate()(calibrators)

    # Next, older Lattice requires 'lattice_sizes'
    # e.g. if you have len(feature_names)=10, you might do [2]*10 => each dimension has 2 vertices
    n_dims = len(feature_names)
    
    # Build the integer monotonicities for Lattice: +1 => increasing, -1 => decreasing, 0 => none
    lattice_monotonicities = []
    for feat in feature_names:
        if feat in custom_monotonic_map:
            if custom_monotonic_map[feat] == 'increasing':
                lattice_monotonicities.append(1)
            elif custom_monotonic_map[feat] == 'decreasing':
                lattice_monotonicities.append(0)
            else:
                lattice_monotonicities.append(0)
        else:
            lattice_monotonicities.append(1)

    # Suppose we want 2 vertices per dimension (2^n total corners).
    # If you have fewer features or need more resolution, you can try [3]*n_dims.
    lattice_out = tfl.layers.Lattice(
        lattice_sizes=[2]*n_dims,
        monotonicities=lattice_monotonicities,
        units=1  # for binary classification
    )(concat_calibrators)

    # Final output => Sigmoid for probability
    outputs = tf.keras.layers.Activation('sigmoid')(lattice_out)
    model = tf.keras.Model(inputs=list(inputs.values()), outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_tf_lattice_model_legacy_sizes(X, y):
    from sklearn.model_selection import train_test_split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    feature_names = X.columns.tolist()

    # Keras Tuner 
    import keras_tuner as kt
    
    # We'll define a custom Brier metric in TF:
    def brier_score_tf(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        return tf.reduce_mean(tf.square(y_true - y_pred))

    train_dict = {col: X_train[col].values for col in feature_names}
    val_dict = {col: X_val[col].values for col in feature_names}

    def model_builder(hp):
        model = build_tf_lattice_custom_monotonic_model_legacy_lattice_sizes(
            hp, feature_names, X_train
        )
        # recompile with brier metric
        model.compile(
            optimizer=model.optimizer,
            loss='binary_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.BinaryCrossentropy(name='log_loss'),
                brier_score_tf
            ]
        )
        return model

    tuner = kt.RandomSearch(
        model_builder,
        objective=kt.Objective('val_brier_score_tf', direction='min'),
        max_trials=5,
        executions_per_trial=1,
        project_name='tf_lattice_sizes',
        overwrite=True
    )

    stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_brier_score_tf', patience=5, mode='min')

    tuner.search(
        train_dict, y_train,
        validation_data=(val_dict, y_val),
        epochs=50,
        batch_size=128,
        callbacks=[stop_early],
        verbose=1
    )

    best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
    best_model = tuner.hypermodel.build(best_hps)
    best_model.fit(
        train_dict, y_train,
        validation_data=(val_dict, y_val),
        epochs=50,
        batch_size=128,
        callbacks=[stop_early],
        verbose=0
    )

    # Evaluate final
    res = best_model.evaluate(val_dict, y_val, verbose=0)
    names = best_model.metrics_names
    res_dict = dict(zip(names, res))
    print("Best TF Lattice hyperparams:", best_hps.values)
    print(f"Brier: {res_dict['brier_score_tf']:.4f}, log_loss: {res_dict['log_loss']:.4f}, acc: {res_dict['accuracy']:.4f}")

    return best_model

In [12]:
train_tf_lattice_model_legacy_sizes(X_train, y_train)

ValueError: Exception encountered when calling layer 'pwl_calibration' (type PWLCalibration).

A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


Call arguments received by layer 'pwl_calibration' (type PWLCalibration):
  • inputs=<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=ORPerc_diff_7_A>